# System Dependencies

To get started with Unstructured.io, we need a few system-wide dependencies:

## Poppler (poppler-utils)
Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

## Tesseract (tesseract-ocr)
Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

## libmagic
File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

In [ ]:
# for linux
!apt-get install poppler-utils tesseract-ocr libmagic-dev

# for mac
# !brew install poppler tesseract libmagic

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
libmagic-dev is already the newest version (1:5.41-3ubuntu0.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.10).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [ ]:
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community langchain-openai
%pip install -Uq python_dotenv

In [ ]:
import json
from typing import List
import os
from google.colab import userdata

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

# Load OpenAI API key from Colab secrets
openai_api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")

    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )

    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "/content/Equation(10087C-v1-basic).pdf"  # Change this to your PDF path
elements = partition_document(file_path)

📄 Partitioning document: /content/Equation(10087C-v1-basic).pdf
✅ Extracted 27 elements


In [ ]:
# elements
# len(elements)


elements

In [ ]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [ ]:
elements[26].to_dict()

{'type': 'UncategorizedText',
 'element_id': '1a6cf1312de2f0b9ad42a0b7ffe0f806',
 'text': 'in',
 'metadata': {'coordinates': {'points': ((np.float64(1403.1388888888885),
     np.float64(1213.823611111111)),
    (np.float64(1403.1388888888885), np.float64(1240.2124999999999)),
    (np.float64(1436.7236111111104), np.float64(1240.2124999999999)),
    (np.float64(1436.7236111111104), np.float64(1213.823611111111))),
   'system': 'PixelSpace',
   'layout_width': 1536,
   'layout_height': 2034},
  'last_modified': '2025-10-03T13:56:48',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': '/content',
  'filename': 'Equation(10087C-v1-basic).pdf',
  'parent_id': '897b3fde7648e7d68cd60947411964ad'}}

In [ ]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

Found 2 images


{'type': 'Image',
 'element_id': '92a0b52ee0ff136c4ab52db8e9daed36',
 'text': 'Rl   lOK   Rl   lOK   75V   R2   lOK   R2   150V -=.   75V  -=- lSOV   lOK   lOK   SOY   SOY   A   B  ',
 'metadata': {'coordinates': {'points': ((np.float64(60.0),
     np.float64(135.99972222222237)),
    (np.float64(60.0), np.float64(520.0000000000001)),
    (np.float64(1416.0002777777777), np.float64(520.0000000000001)),
    (np.float64(1416.0002777777777), np.float64(135.99972222222237))),
   'system': 'PixelSpace',
   'layout_width': 1536,
   'layout_height': 2034},
  'last_modified': '2025-10-03T13:56:48',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': '/content',
  'filename': 'Equation(10087C-v1-basic).pdf'}}

In [ ]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html


Found 1 tables


{'type': 'Table',
 'element_id': '680b4eedeca42802b36482e7e1af0fe3',
 'text': 'ohms Sensitivity = --- = volts 1 1 --::-:-- = --- ohms',
 'metadata': {'detection_class_prob': 0.6524971127510071,
  'coordinates': {'points': ((np.float64(100.20901489257812),
     np.float64(1031.8870849609375)),
    (np.float64(100.20901489257812), np.float64(1122.1810302734375)),
    (np.float64(651.9202880859375), np.float64(1122.1810302734375)),
    (np.float64(651.9202880859375), np.float64(1031.8870849609375))),
   'system': 'PixelSpace',
   'layout_width': 1536,
   'layout_height': 2034},
  'last_modified': '2025-10-03T13:56:48',
  'text_as_html': '<table><tbody><tr><td>Sensitivity</td><td>= _ = volts</td><td>volts.</td><td>__1 amperes</td></tr></tbody></table>',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': '/content',
  'filename': 'Equation(10087C-v1-basic).pdf'}}

In [ ]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")

    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )

    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 2 chunks


In [ ]:
# View all chunks
# chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>"}

In [ ]:
# View a single chunk
# chunks[2].to_dict()

# View original elements
chunks[1].metadata.orig_elements[-1].to_dict()
# Note: 4th chunk has the first image + 11th chunk has the first table in the sample PDF

{'type': 'UncategorizedText',
 'element_id': '1a6cf1312de2f0b9ad42a0b7ffe0f806',
 'text': 'in',
 'metadata': {'coordinates': {'points': ((np.float64(1403.1388888888885),
     np.float64(1213.823611111111)),
    (np.float64(1403.1388888888885), np.float64(1240.2124999999999)),
    (np.float64(1436.7236111111104), np.float64(1240.2124999999999)),
    (np.float64(1436.7236111111104), np.float64(1213.823611111111))),
   'system': 'PixelSpace',
   'layout_width': 1536,
   'layout_height': 2034},
  'last_modified': '2025-10-03T13:56:48',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': '/content',
  'filename': 'Equation(10087C-v1-basic).pdf',
  'parent_id': '897b3fde7648e7d68cd60947411964ad'}}

In [ ]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }

    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)

            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)

    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""

    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOpenAI(model="gpt-4o", temperature=0)

        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """

        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]

        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })

        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])

        return response.content

    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")

    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")

        # Analyze chunk content
        content_data = separate_content_types(chunk)

        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']

        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )

        langchain_documents.append(doc)

    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/2
     Types found: ['text', 'table', 'image']
     Tables: 1, Images: 2
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: **SEARCHABLE DESCRIPTION:**

**Key Facts, Numbers, and Data Points:**
- The document discusses the sensitivity of voltmeters, defined as ohms per volt (Ω/V).
- Sensitivity is calculated by dividing th...
   Processing chunk 2/2
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
✅ Processed 2 chunks


In [ ]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "BASIC ELECTRONICS VOLUl\'t\'lE I\\n\\nRl lOK Rl lOK 75V R2 lOK R2 150V -=. 75V -=- lSOV lOK lOK SOY SOY A B\\n\\nA\\n\\nB\\n\\n179.10 Figure 3-11. -Shunting action caused by a voltmeter.\\n\\nSensitivity\\n\\nThe sensitivity of voltmeter is given in a ohms per volt, (0/E), and may be determined by dividing the resistance, Rm , of the meter plus the series resistance R5, by the full scale reading in volts. Thus, sensitivity = CRm + R5)/E. This saying that the sensitivity in equation form, is the same as is equal to the reciprocal of the full scale deflection current. In equation form then,\\n\\n1\\n\\nohms Sensitivity = --- = volts 1 1 --::-:-- = --- ohms\\n\\nvolts amperes\\n\\nThus,. the sensitivity of a 100 microampere movement is the reciprocal of .0001 ampere, or 10,000 ohms per volt.\\n\\nfew megohms. The megger is widely used for measuring insulation resistance, such as between a wire and the outer surface of its insulation, 

In [ ]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []

    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)

    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 2 chunks to chunks_export.json


In [ ]:
import os

def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")

    # Access the API key from the environment variable
    openai_api_key = os.environ.get("OPENAI_API_KEY")
    if not openai_api_key:
        raise ValueError("OPENAI_API_KEY environment variable not set.")

    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", api_key=openai_api_key)

    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")

    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


In [ ]:
import os

openai_api_key = os.environ.get("OPENAI_API_KEY")

if openai_api_key:
    print("OPENAI_API_KEY environment variable is set.")
else:
    print("OPENAI_API_KEY environment variable is NOT set.")

OPENAI_API_KEY environment variable is set.


In [ ]:
# Access the API key from the environment variable
openai_api_key = os.environ.get("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set.")

In [ ]:
import os

def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")

    # Access the API key from the environment variable
    openai_api_key = os.environ.get("OPENAI_API_KEY")
    if not openai_api_key:
        raise ValueError("OPENAI_API_KEY environment variable not set.")

    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", api_key=openai_api_key)

    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")

    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


In [ ]:
# After your retrieval
query = "Explain the sensitivity of voltmeter "
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

✅ Exported 3 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': "BASIC ELECTRONICS VOLUl't'lE I\n\nRl lOK Rl lOK 75V R2 lOK R2 150V -=. 75V -=- lSOV lOK lOK SOY SOY A B\n\nA\n\nB\n\n179.10 Figure 3-11. -Shunting action caused by a voltmeter.\n\nSensitivity\n\nThe sensitivity of voltmeter is given in a ohms per volt, (0/E), and may be determined by dividing the resistance, Rm... [Contains 1 table(s)] [Contains 2 image(s)]",
  'metadata': {'original_content': {'raw_text': "BASIC ELECTRONICS VOLUl't'lE I\n\nRl lOK Rl lOK 75V R2 lOK R2 150V -=. 75V -=- lSOV lOK lOK SOY SOY A B\n\nA\n\nB\n\n179.10 Figure 3-11. -Shunting action caused by a voltmeter.\n\nSensitivity\n\nThe sensitivity of voltmeter is given in a ohms per volt, (0/E), and may be determined by dividing the resistance, Rm , of the meter plus the series resistance R5, by the full scale reading in volts. Thus, sensitivity = CRm + R5)/E. This saying that the sensitivity in equation form, is the same as is equal to the reciprocal of the full scale deflection

In [ ]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)

    # Step 1: Partition
    elements = partition_document(pdf_path)

    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)

    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)

    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2/chroma_db")

    print("🎉 Pipeline completed successfully!")
    return db

# Run the complete pipeline

In [ ]:
db = run_complete_ingestion_pipeline("/content/Equation(10087C-v1-basic).pdf")

🚀 Starting RAG Ingestion Pipeline
📄 Partitioning document: /content/Equation(10087C-v1-basic).pdf
✅ Extracted 27 elements
🔨 Creating smart chunks...
✅ Created 2 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/2
     Types found: ['text', 'table', 'image']
     Tables: 1, Images: 2
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: **SEARCHABLE DESCRIPTION:**

**Key Facts, Numbers, and Data Points:**
- The document discusses the sensitivity of voltmeters, defined as ohms per volt (Ω/V).
- Sensitivity is calculated by dividing th...
   Processing chunk 2/2
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
✅ Processed 2 chunks
🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv2/chroma_db
🎉 Pipeline completed successfully!


In [ ]:
# Query the vector store
query = "Explain the definition of sensitivity of voltmeter including equation in the result."

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""

    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOpenAI(model="gpt-4o", temperature=0)

        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""

        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"

            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])

                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"

                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"

            prompt_text += "\n"

        prompt_text += """
Please provide a clear, comprehensive answer using the text, symbols, equations, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]

        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])

                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })

        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])

        return response.content

    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

The sensitivity of a voltmeter is expressed in ohms per volt and is calculated by dividing the total resistance of the meter (Rm) plus any series resistance (Rs) by the full-scale reading in volts (E). The formula is:

\[ \text{Sensitivity} = \frac{(R_m + R_s)}{E} \]

This is equivalent to the reciprocal of the full-scale deflection current. In equation form:

\[ \text{Sensitivity} = \frac{1}{\text{Full-scale current in amperes}} \]

For example, if the full-scale deflection current is 100 microamperes (0.0001 amperes), the sensitivity would be:

\[ \text{Sensitivity} = \frac{1}{0.0001} = 10,000 \text{ ohms per volt} \]

This means the voltmeter has a sensitivity of 10,000 ohms per volt.


In [ ]:
import os

# Replace "YOUR_OPENAI_API_KEY" with your actual OpenAI API key
os.environ["OPENAI_API_KEY"] = "replace with your own API key"